In [1]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
import PIL 
import os
import cv2
import pathlib
import tensorflow_hub as hub
import tf_keras as tfk 

In [15]:
IMAGE_SHAPE = (128,128)
classifier = tfk.Sequential([
    hub.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-128-classification/2",
                   input_shape=IMAGE_SHAPE+(3,))
]) 

In [3]:
# dataset_url = "http://download.tensorflow.org/example_images/flower_photos.tgz"
# data_dir = tf.keras.utils.get_file('flower_photos_4',origin=dataset_url,cache_dir='.',untar=True)

In [4]:
data_dir = './datasets/flower_photos_4/flower_photos'

In [5]:
data_dir = pathlib.Path(data_dir)
data_dir

WindowsPath('datasets/flower_photos_4/flower_photos')

In [6]:
flowers_images_dict = {
    "roses":list(data_dir.glob("roses/*")),
    "tulips":list(data_dir.glob("tulips/*")),
    "sunflowers":list(data_dir.glob("sunflowers/*")),
    "dandelion":list(data_dir.glob("dandelion/*")),
    "daisy":list(data_dir.glob("daisy/*"))
}

flowers_labels_dict = {
    "roses":0,
    "tulips":1,
    "sunflowers":2,
    "dandelion":3,
    "daisy":4
}

In [7]:
X,y = [],[]

for flower_name,images in flowers_images_dict.items():
    for image in images:
        img = cv2.imread(str(image))
        resize_img = cv2.resize(img,IMAGE_SHAPE)
        X.append(resize_img)
        y.append(flowers_labels_dict[flower_name])

In [8]:
X = np.array(X)/255
y = np.array(y)

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [11]:
predtrained_model_without_top_layer = tfk.Sequential([
  hub.KerasLayer("https://www.kaggle.com/models/google/mobilenet-v2/TensorFlow2/035-128-feature-vector/2",
                 input_shape=IMAGE_SHAPE+(3,), trainable=False)
])

model = tfk.Sequential([
    predtrained_model_without_top_layer,
    tfk.layers.Dense(100,'relu'),
    tfk.layers.Dense(5,'sigmoid')
])
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 dense (Dense)               (None, 100)               128100    
                                                                 
 dense_1 (Dense)             (None, 5)                 505       
                                                                 
Total params: 538813 (2.06 MB)
Trainable params: 128605 (502.36 KB)
Non-trainable params: 410208 (1.56 MB)
_________________________________________________________________


In [16]:
model.compile(
    optimizer='adam',
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
model.fit(X_train,y_train,epochs=5)

Epoch 1/5
77/77 [==============================] - 9s 83ms/step - loss: 0.0990 - accuracy: 0.9744
Epoch 2/5
77/77 [==============================] - 6s 82ms/step - loss: 0.0460 - accuracy: 0.9947
Epoch 3/5
77/77 [==============================] - 7s 90ms/step - loss: 0.0211 - accuracy: 0.9984
Epoch 4/5
77/77 [==============================] - 9s 113ms/step - loss: 0.0109 - accuracy: 1.0000
Epoch 5/5
77/77 [==============================] - 11s 146ms/step - loss: 0.0066 - accuracy: 1.0000


In [20]:
model.evaluate(X_test,y_test)

38/38 [==============================] - 3s 81ms/step - loss: 0.6611 - accuracy: 0.8342


[0.6610885858535767, 0.8341584205627441]